# **Script em Python de Tratamento de Dados**

## Objetivo

Este script tem como objetivo realizar o processo de tratamento dos datasets utilizados no teste técnico de Analista de Dados. O script automatiza a etapa de carregamento, análise e limpeza dos dados, garantindo que os arquivos estejam prontos para análise ou utilização em ferramentas de Business Intelligence.

## Funcionalidades do Script

* Importação automática dos arquivos de dados.
* Carregamento dos datasets utilizando a biblioteca Pandas.
* Verificação da estrutura dos dados (tipos de colunas e quantidade de registros).
* Identificação de valores ausentes (missing values).
* Identificação de registros duplicados.
* Remoção de duplicidades nos datasets de vendas.
* Unificação dos datasets **Vendas.xlsx** e **Vendas_2T.xlsx** em um único dataset consolidado chamado **vendas_total**, garantindo uma base única de vendas.
* Remoção das tabelas intermediárias de vendas após a consolidação, evitando duplicidade de exportação.
* Padronização e tradução dos nomes das colunas dos datasets para o idioma português, facilitando a leitura e a modelagem de dados.
* Organização dos datasets tratados em uma estrutura única.
* Exportação dos datasets tratados para arquivos no formato CSV.

## Resultado Final

Ao final da execução, o script gera uma pasta chamada **datasets_tratados** contendo os datasets já limpos, padronizados e preparados para análise.

Os arquivos exportados são:

* consultores_tratado.csv
* lojas_tratado.csv
* metas_tratado.csv
* vendas_total_tratado.csv

O dataset **vendas_total_tratado.csv** representa a consolidação dos dados de vendas provenientes dos arquivos **Vendas.xlsx** e **Vendas_2T.xlsx**.

## Tecnologias utilizadas

* Python
* Pandas
* Manipulação de arquivos


In [7]:
import pandas as pd
import os

# ================================
# 1. Caminho dos arquivos
# ================================

caminho_arquivos = {
    "metas": "Metas.xlsx",
    "lojas": "Lojas.xlsx",
    "vendas": "Vendas.xlsx",
    "vendas_2t": "Vendas_2T.xlsx",
    "consultores": "Consultores.xlsx"
}

# ================================
# 2. Carregar os arquivos
# ================================

dataframes = {}

for nome, caminho in caminho_arquivos.items():
    df = pd.read_excel(caminho)
    dataframes[nome] = df
    print(f"Arquivo {nome} carregado com sucesso")

# ================================
# 3. Informações dos datasets
# ================================

for nome, df in dataframes.items():
    print(f"\nInformações do dataset: {nome}")
    print(df.info())

# ================================
# 4. Verificar dados faltantes
# ================================

print("\nVerificando dados faltantes")

for nome, df in dataframes.items():

    missing_counts = df.isnull().sum()
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    missing_info = pd.DataFrame({
        "contagem_faltante": missing_counts,
        "porcentagem_faltante": missing_percentage
    })

    missing_info = missing_info[missing_info["contagem_faltante"] > 0]

    print(f"\nDados faltantes em {nome}")

    if missing_info.empty:
        print("Nenhum dado faltante")
    else:
        print(missing_info)

# ================================
# 5. Verificar duplicados
# ================================

print("\nVerificando duplicados")

for nome, df in dataframes.items():

    duplicados = df.duplicated().sum()

    print(f"{nome}: {duplicados} linhas duplicadas")

# ================================
# 6. Remover duplicados
# ================================

dataframes["vendas"] = dataframes["vendas"].drop_duplicates()
dataframes["vendas_2t"] = dataframes["vendas_2t"].drop_duplicates()

print("\nDuplicados removidos de vendas e vendas_2t")

# ================================
# 7. Unificar tabelas de vendas
# ================================

vendas_total = pd.concat(
    [dataframes["vendas"], dataframes["vendas_2t"]],
    ignore_index=True
)

vendas_total = vendas_total.drop_duplicates()

dataframes["vendas_total"] = vendas_total

print("\nTabelas vendas e vendas_2t unificadas com sucesso")
print(f"Total de registros: {len(vendas_total)}")

del dataframes["vendas"]
del dataframes["vendas_2t"]

# ================================
# 8. Traduzir nomes das colunas
# ================================

traducao_colunas = {
    "IdStore": "id_loja",
    "Store": "loja",
    "StoreName": "nome_loja",
    "Date": "data",
    "Product": "produto",
    "Category": "categoria",
    "Quantity": "quantidade",
    "Price": "preco",
    "Revenue": "faturamento",
    "Consultant": "consultor",
    "ConsultantID": "id_consultor",
    "Goal": "meta"
}

for nome, df in dataframes.items():
    df.rename(columns=traducao_colunas, inplace=True)

print("\nColunas traduzidas para português com sucesso!")

# ================================
# 9. Criar pasta de saída
# ================================

os.makedirs("datasets_tratados", exist_ok=True)

# ================================
# 10. Exportar datasets tratados
# ================================

for nome, df in dataframes.items():

    caminho_saida = f"datasets_tratados/{nome}_tratado.csv"

    df.to_csv(caminho_saida, index=False)

    print(f"Arquivo exportado: {caminho_saida}")

print("\nTodos os datasets foram exportados com sucesso!")

Arquivo metas carregado com sucesso
Arquivo lojas carregado com sucesso
Arquivo vendas carregado com sucesso
Arquivo vendas_2t carregado com sucesso
Arquivo consultores carregado com sucesso

Informações do dataset: metas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1222 entries, 0 to 1221
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   IdStore        1222 non-null   int64         
 1   Date           1222 non-null   datetime64[ns]
 2   Week           1222 non-null   int64         
 3   RevenueTarget  1222 non-null   int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 38.3 KB
None

Informações do dataset: lojas
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   IdStore  47 non-null     int64 
 1   Store    47 non-null     object
 2   Size     47 non-n